# FLO CLTV — Keşifsel Veri Analizi (EDA)

Amaç: Modele girmeden önce veriyi tanımak, aykırı değerleri görmek,
müşteri davranışlarını anlamak.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from src.data_loader import load_data, preprocess

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")
sns.set_theme(style="whitegrid")

with open("../config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

## 1. Ham Veri

In [ ]:
df_raw = load_data(config["data"]["raw_path"])
print(f"Boyut: {df_raw.shape}")
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe().T

In [ ]:
print("Eksik değerler:")
df_raw.isnull().sum()

## 2. Ön İşleme Sonrası Veri

In [ ]:
df = preprocess(df_raw, outlier_cols=config["outlier_cols"])
df.describe().T

## 3. Alışveriş Kanalı Analizi

In [ ]:
channel = df.groupby("order_channel").agg(
    musteri_sayisi=("master_id", "count"),
    toplam_siparis=("order_num_total", "sum"),
    toplam_harcama=("customer_value_total", "sum"),
    ort_harcama=("customer_value_total", "mean"),
    ort_siparis=("order_num_total", "mean"),
).round(2)
channel

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

channel["musteri_sayisi"].plot(kind="bar", ax=axes[0], color="steelblue", rot=20)
axes[0].set_title("Müşteri Sayısı")
axes[0].set_ylabel("Adet")

channel["ort_siparis"].plot(kind="bar", ax=axes[1], color="seagreen", rot=20)
axes[1].set_title("Ortalama Sipariş Sayısı")

channel["ort_harcama"].plot(kind="bar", ax=axes[2], color="coral", rot=20)
axes[2].set_title("Ortalama Harcama (₺)")

fig.suptitle("Kanal Bazında Özet", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Sayısal Değişken Dağılımları

In [ ]:
num_cols = [
    "order_num_total_ever_online",
    "order_num_total_ever_offline",
    "customer_value_total_ever_online",
    "customer_value_total_ever_offline",
    "order_num_total",
    "customer_value_total",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    ax.hist(df[col], bins=50, color="steelblue", edgecolor="white", linewidth=0.4)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("Frekans")

fig.suptitle("Sayısal Değişken Dağılımları (Aykırı Değer Baskılandıktan Sonra)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Aykırı Değer — Önce / Sonra Karşılaştırması

In [ ]:
from src.data_loader import outlier_thresholds

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, config["outlier_cols"]):
    low, up = outlier_thresholds(df_raw, col)
    ax.boxplot(df_raw[col].dropna(), vert=False, patch_artist=True,
               boxprops=dict(facecolor="lightcoral"))
    ax.axvline(up, color="red", linestyle="--", linewidth=1.2, label=f"Üst eşik: {up:.0f}")
    ax.axvline(low, color="blue", linestyle="--", linewidth=1.2, label=f"Alt eşik: {low:.0f}")
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle("Aykırı Değer Eşikleri (Ham Veri)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Müşteri Yaşı ve Satın Alma Sıklığı

In [ ]:
import datetime as dt

analysis_date = pd.to_datetime(config["analysis"]["analysis_date"])

df["customer_age_days"] = (analysis_date - df["first_order_date"]).dt.days
df["recency_days"] = (analysis_date - df["last_order_date"]).dt.days

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df["customer_age_days"], bins=50, color="mediumpurple", edgecolor="white", linewidth=0.4)
axes[0].set_title("Müşteri Yaşı (Gün — İlk Alışverişten Bu Yana)")
axes[0].set_xlabel("Gün")
axes[0].set_ylabel("Müşteri Sayısı")

axes[1].hist(df["recency_days"], bins=50, color="darkorange", edgecolor="white", linewidth=0.4)
axes[1].set_title("Recency (Son Alışverişten Bu Yana Geçen Gün)")
axes[1].set_xlabel("Gün")

plt.tight_layout()
plt.show()

## 7. Harcama ve Sipariş İlişkisi

In [ ]:
sample = df.sample(2000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(sample["order_num_total"], sample["customer_value_total"],
                alpha=0.3, color="steelblue", s=15)
axes[0].set_title("Toplam Sipariş vs Toplam Harcama")
axes[0].set_xlabel("Sipariş Sayısı")
axes[0].set_ylabel("Harcama (₺)")

axes[1].scatter(sample["order_num_total"],
                sample["customer_value_total"] / sample["order_num_total"],
                alpha=0.3, color="seagreen", s=15)
axes[1].set_title("Sipariş Sayısı vs Ortalama Sepet Değeri")
axes[1].set_xlabel("Sipariş Sayısı")
axes[1].set_ylabel("Ortalama Sepet (₺)")

plt.tight_layout()
plt.show()

## 8. İlgi Kategorisi Analizi

In [ ]:
from collections import Counter

all_cats = df["interested_in_categories_12"].dropna()
cat_counts = Counter()

for row in all_cats:
    cats = row.strip("[]").replace(" ", "").split(",")
    cat_counts.update(cats)

cat_df = pd.DataFrame(cat_counts.items(), columns=["kategori", "sayi"]).sort_values("sayi", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(cat_df["kategori"], cat_df["sayi"], color=sns.color_palette("Set2", len(cat_df)))
ax.set_title("Son 12 Ayda Kategori Bazında Müşteri Sayısı", fontsize=13, fontweight="bold")
ax.set_ylabel("Müşteri Sayısı")
ax.set_xticklabels(cat_df["kategori"], rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 9. CLTV Model Girdileri Dağılımı

Aşağıdaki dağılımlar modele girecek olan değişkenleri gösterir.
Frequency > 1 filtresi uygulanmıştır.

In [ ]:
from src.cltv import build_cltv_dataframe

cltv_df = build_cltv_dataframe(df, analysis_date=config["analysis"]["analysis_date"])

cltv_df[["recency_cltv_weekly", "T_weekly", "frequency", "monetary_cltv_avg"]].describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

cols = {
    "recency_cltv_weekly": ("Recency (Hafta)", "steelblue"),
    "T_weekly": ("Müşteri Yaşı (Hafta)", "mediumpurple"),
    "frequency": ("Frequency (Sipariş)", "seagreen"),
    "monetary_cltv_avg": ("Monetary Avg (₺)", "coral"),
}

for ax, (col, (label, color)) in zip(axes, cols.items()):
    ax.hist(cltv_df[col], bins=50, color=color, edgecolor="white", linewidth=0.4)
    ax.set_title(label, fontsize=11)
    ax.set_ylabel("Müşteri Sayısı")

fig.suptitle("CLTV Model Girdileri Dağılımı (frequency > 1)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Korelasyon Matrisi

In [ ]:
corr = cltv_df[["recency_cltv_weekly", "T_weekly", "frequency", "monetary_cltv_avg"]].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Model Girdileri Korelasyon Matrisi", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()